In [ ]:
import numpy as np
from scipy.optimize import curve_fit
from scipy.special import i0  # Modified Bessel function of first kind, order 0
import matplotlib.pyplot as plt
from pathlib import Path
import flammkuchen as fl

In [ ]:
def extract_responses(neural_data, stimulus_data, response_window=5):
    """
    Extract neural responses for each stimulus position
    
    Parameters:
    -----------
    neural_data : array-like
        Shape (n_timepoints, n_neurons) containing calcium signals
    stimulus_data : array-like
        Shape (8, n_timepoints) containing stimulus presentations
    response_window : int
        Number of timepoints to average after stimulus onset
        
    Returns:
    --------
    responses : array-like
        Shape (n_neurons, 8) containing average responses for each position
    """
    n_positions = stimulus_data.shape[0]
    n_neurons = neural_data.shape[1]
    responses = np.zeros((n_neurons, n_positions))
    
    for pos in range(n_positions):
        # Find timepoints where this position was presented
        stim_times = np.where(stimulus_data[pos, :] > 0)[0]
        
        # For each stimulus presentation
        pos_responses = []
        for t in stim_times:
            # Extract response window (checking we don't go past the end)
            if t + response_window <= neural_data.shape[0]:
                window_response = neural_data[t:t+response_window, :].mean(axis=0)
                pos_responses.append(window_response)
        
        # Average across all presentations of this position
        if pos_responses:
            responses[:, pos] = np.mean(pos_responses, axis=0)
    
    return responses

In [ ]:
def von_mises(x, amp, kappa, mu, offset):
    """
    Von Mises function (circular normal distribution)
    
    Parameters:
    -----------
    x : array-like
        Angles in radians
    amp : float
        Amplitude
    kappa : float
        Concentration parameter
    mu : float
        Preferred direction (radians)
    offset : float
        Baseline offset
    """
    return amp * np.exp(kappa * np.cos(x - mu)) / (2 * np.pi * i0(kappa)) + offset

def fit_tuning_curve(angles, responses, max_kappa=20):
    """
    Fit von Mises function to neural responses with parameter constraints
    
    Parameters:
    -----------
    angles : array-like
        Stimulus angles in degrees
    responses : array-like
        Neural responses for each angle
    max_kappa : float
        Maximum allowed value for kappa parameter
    
    Returns:
    --------
    params : tuple
        Fitted parameters (amplitude, kappa, mu, offset)
    r_squared : float
        R-squared value of the fit
    """
    # Convert angles to radians
    angles_rad = np.deg2rad(angles)
    
    # Normalize responses to 0-1 range for better fitting
    responses_norm = (responses - np.min(responses)) / (np.max(responses) - np.min(responses))
    
    # Initial parameter guesses
    p0 = [
        1.0,               # amplitude (normalized)
        1.0,               # kappa
        angles_rad[np.argmax(responses_norm)],  # mu
        0.0                # offset (normalized)
    ]
    
    # Parameter bounds (lower, upper)
    bounds = (
        [0.0, 0.0, -np.pi, 0.0],           # lower bounds
        [2.0, max_kappa, np.pi, 1.0]        # upper bounds
    )
    
    try:
        # Fit with bounds
        params, pcov = curve_fit(von_mises, angles_rad, responses_norm, 
                               p0=p0, bounds=bounds)
        
        # Calculate R-squared
        y_fit = von_mises(angles_rad, *params)
        ss_res = np.sum((responses_norm - y_fit) ** 2)
        ss_tot = np.sum((responses_norm - np.mean(responses_norm)) ** 2)
        r_squared = 1 - (ss_res / ss_tot)
        
        # Denormalize parameters
        response_range = np.max(responses) - np.min(responses)
        params[0] *= response_range
        params[3] = params[3] * response_range + np.min(responses)
        
        # Validate fit quality
        is_valid, diagnostics = validate_fit_quality(angles, responses, params, r_squared)
        
        if is_valid:
            return params, r_squared, diagnostics
        else:
            return None, None, diagnostics
            
    except:
        return None, None, None


def analyze_neural_data(neural_data, stimulus_data, angles, min_r_squared=0.3):
    """
    Analyze population of neurons and create tuning curves
    
    Parameters:
    -----------
    neural_data : array-like
        Shape (n_timepoints, n_neurons) containing calcium signals
    stimulus_data : array-like
        Shape (8, n_timepoints) containing stimulus presentations
    angles : array-like
        Stimulus angles in degrees
    min_r_squared : float
        Minimum R-squared value for accepting a fit
    
    Returns:
    --------
    kappas : list
        Concentration parameters for all successfully fitted neurons
    all_fits : list
        All fitted parameters for each neuron
    responses : array-like
        Shape (n_neurons, 8) containing average responses for each position
    r_squared_values : list
        R-squared values for all accepted fits
    all_diagnostics : list
        Diagnostic information for all accepted fits
    """
    # Extract responses for each position
    responses = extract_responses(neural_data, stimulus_data)
    
    kappas = []
    all_fits = []
    r_squared_values = []
    all_diagnostics = []
    
    for neuron_idx in range(responses.shape[0]):
        neuron_responses = responses[neuron_idx, :]
        
        # Fit von Mises function
        params, r_squared, diagnostics = fit_tuning_curve(angles, neuron_responses)  # Now unpacking three values
        
        if params is not None and r_squared is not None and r_squared >= min_r_squared:
            kappas.append(params[1])
            all_fits.append(params)
            r_squared_values.append(r_squared)
            all_diagnostics.append(diagnostics)
    
    return kappas, all_fits, responses, r_squared_values, all_diagnostics
    

def plot_results(responses, angles, kappas, all_fits, r_squared_values):
    """
    Create visualization of results
    
    Parameters:
    -----------
    responses : array-like
        Shape (n_neurons, 8) containing average responses for each position
    angles : array-like
        Stimulus angles in degrees
    kappas : list
        Fitted kappa values
    all_fits : list
        All fitted parameters
    """
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 3))
    
    # Plot example tuning curves for first few neurons
    angles_rad = np.deg2rad(angles)
    x_fit = np.linspace(np.deg2rad(-60), np.deg2rad(60), 100)
    
    for i in range(min(3, len(all_fits))):
        neuron_responses = responses[i, :]
        params = all_fits[i]
        
        # Plot raw data
        ax1.scatter(angles_rad, neuron_responses, alpha=0.5, label=f'Neuron {i+1}')
        
        # Plot fitted curve
        y_fit = von_mises(x_fit, *params)
        ax1.plot(x_fit, y_fit)
    
    ax1.set_xlabel('Angle (radians)')
    ax1.set_ylabel('Response')
    ax1.set_title('Example Tuning Curves')
    ax1.legend()
    
    # Plot kappa distribution on polar plot
    ax2 = plt.subplot(122, projection='polar')
    
    # Convert preferred directions (mu) to angles for plotting
    preferred_directions = [params[2] for params in all_fits]  # mu is the third parameter
    
    # Create scatter plot only within the -60 to +60 degree range
    ax2.scatter(preferred_directions, kappas)
    
    # Set the limits of the polar plot to only show the relevant angle range
    ax2.set_thetamin(-60)
    ax2.set_thetamax(60)
    
    ax2.set_title('Kappa Values vs Preferred Direction')
    ax2.set_rticks([0, max(kappas)/2, max(kappas)])
    
    # Add summary statistics to the plot
    plt.figtext(0.02, 0.02, 
                f'n = {len(kappas)} neurons\n'
                f'mean κ = {np.mean(kappas):.2f} ± {np.std(kappas):.2f}\n'
                f'mean R² = {np.mean(r_squared_values):.2f}',
                fontsize=8)
    
    plt.tight_layout()
    plt.show()

    

In [ ]:
def plot_kappa_examples(responses, angles, all_fits, r_squared_values, kappa_ranges=[(0, 5), (5, 10), (10, 15), (15, None)]):
    """
    Plot example neurons from different kappa ranges
    
    Parameters:
    -----------
    responses : array-like
        Shape (n_neurons, 8) containing average responses for each position
    angles : array-like
        Stimulus angles in degrees
    all_fits : list
        All fitted parameters for each neuron
    r_squared_values : list
        R-squared values for all fits
    kappa_ranges : list of tuples
        Ranges of kappa values to examine
    """
    n_ranges = len(kappa_ranges)
    fig, axes = plt.subplots(n_ranges, 3, figsize=(15, 4*n_ranges))
    
    angles_rad = np.deg2rad(angles)
    x_fit = np.linspace(np.deg2rad(-60), np.deg2rad(60), 100)
    
    for i, (kmin, kmax) in enumerate(kappa_ranges):
        # Find neurons in this kappa range
        range_neurons = []
        for j, params in enumerate(all_fits):
            kappa = params[1]
            if kmin <= kappa and (kmax is None or kappa < kmax):
                range_neurons.append((j, kappa, r_squared_values[j]))
        
        if range_neurons:
            # Sort by R-squared and take the best 3 examples
            range_neurons.sort(key=lambda x: x[2], reverse=True)
            examples = range_neurons[:3]
            
            for j, (neuron_idx, kappa, r2) in enumerate(examples):
                neuron_responses = responses[neuron_idx, :]
                params = all_fits[neuron_idx]
                
                # Plot raw data and fit
                axes[i, j].scatter(angles_rad, neuron_responses, alpha=0.5, color='blue')
                y_fit = von_mises(x_fit, *params)
                axes[i, j].plot(x_fit, y_fit, 'r-')
                
                axes[i, j].set_title(f'κ={kappa:.2f}, R²={r2:.2f}\n' +
                                    f'SNR={all_diagnostics[neuron_idx]["signal_to_noise"]:.1f}, ' +
                                    f'Peak Δ={all_diagnostics[neuron_idx]["peak_angle_diff"]:.1f}°')

                # Add labels
                axes[i, j].set_xlabel('Angle (radians)')
                axes[i, j].set_ylabel('Response')
                
                # Add grid for better readability
                axes[i, j].grid(True, alpha=0.3)
        else:
            for j in range(3):
                axes[i, j].text(0.5, 0.5, 'No neurons in this range', 
                              ha='center', va='center', transform=axes[i, j].transAxes)
        
        # Add range label
        range_label = f'κ ∈ [{kmin}, {kmax if kmax else "∞"})'
        fig.text(0.01, 1 - (i + 0.5) / n_ranges, range_label, 
                va='center', ha='left', fontsize=12)
    
    plt.tight_layout()
    plt.show()

def plot_neuron_details(neural_data, stimulus_data, neuron_idx, response_window=5):
    """
    Plot detailed view of a single neuron's responses
    
    Parameters:
    -----------
    neural_data : array-like
        Shape (n_timepoints, n_neurons) containing calcium signals
    stimulus_data : array-like
        Shape (8, n_timepoints) containing stimulus presentations
    neuron_idx : int
        Index of the neuron to examine
    response_window : int
        Number of timepoints to show after stimulus onset
    """
    n_positions = stimulus_data.shape[0]
    
    # Create figure with subplots for each position
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    axes = axes.flatten()
    
    # Get this neuron's trace
    neuron_trace = neural_data[:, neuron_idx]
    
    for pos in range(n_positions):
        ax = axes[pos]
        
        # Find stimulus presentations for this position
        stim_times = np.where(stimulus_data[pos, :] > 0)[0]
        
        # Plot individual responses
        for t in stim_times[:10]:  # Plot up to 10 trials
            if t + response_window <= len(neuron_trace):
                window = slice(t-5, t+response_window)  # Include 5 timepoints before stimulus
                ax.plot(range(-5, response_window), 
                       neuron_trace[window], 
                       alpha=0.3, 
                       color='gray')
        
        # Plot mean response
        mean_response = np.mean([neuron_trace[t-5:t+response_window] 
                               for t in stim_times 
                               if t + response_window <= len(neuron_trace)], 
                              axis=0)
        ax.plot(range(-5, response_window), mean_response, 'r-', linewidth=2)
        
        ax.axvline(x=0, color='k', linestyle='--', alpha=0.5)  # Mark stimulus onset
        ax.set_title(f'Position {pos} ({angles[pos]}°)')
        ax.grid(True, alpha=0.3)
    
    plt.suptitle(f'Neuron {neuron_idx} Responses', fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
def validate_fit_quality(angles, responses, params, r_squared):
    """
    Validate the quality of von Mises fit beyond R-squared
    
    Parameters:
    -----------
    angles : array-like
        Stimulus angles in degrees
    responses : array-like
        Neural responses for each angle
    params : tuple
        Fitted parameters (amplitude, kappa, mu, offset)
    r_squared : float
        R-squared value of the fit
        
    Returns:
    --------
    is_valid : bool
        Whether the fit passes quality criteria
    diagnostics : dict
        Dictionary containing diagnostic measures
    """
    angles_rad = np.deg2rad(angles)
    y_fit = von_mises(angles_rad, *params)
    
    # Calculate various quality metrics
    diagnostics = {}
    
    # 1. Check if peak of fitted curve aligns with maximum response
    peak_angle_data = angles[np.argmax(responses)]
    peak_angle_fit = np.rad2deg(params[2])  # mu parameter
    peak_diff = abs(peak_angle_data - peak_angle_fit)
    diagnostics['peak_angle_diff'] = peak_diff
    
    # 2. Check response dynamic range
    response_range = np.ptp(responses)
    noise_level = np.std(responses - y_fit)
    signal_to_noise = response_range / noise_level if noise_level > 0 else 0
    diagnostics['signal_to_noise'] = signal_to_noise
    
    # 3. Check for overfitting - compare adjacent points
    diffs = np.abs(np.diff(responses))
    mean_diff = np.mean(diffs)
    diagnostics['mean_adjacent_diff'] = mean_diff
    
    # More lenient criteria for a good fit
    is_valid = (
        peak_diff < 60 and           # Was 30, now 45 degrees
        signal_to_noise > 1.5 and    # Was 2, now 1.5
        mean_diff < response_range * 0.8  # Was 0.5, now 0.8
    )
    
    return is_valid, diagnostics

    

def plot_results(responses, angles, kappas, all_fits, all_diagnostics):
    """
    Create visualization of results
    
    Parameters:
    -----------
    responses : array-like
        Shape (n_neurons, 8) containing average responses for each position
    angles : array-like
        Stimulus angles in degrees
    kappas : list
        Fitted kappa values
    all_fits : list
        All fitted parameters for each neuron
    all_diagnostics : list
        Diagnostic information for all accepted fits
    """
    # Create figure with two subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    
    # Plot example tuning curves for first few neurons
    angles_rad = np.deg2rad(angles)
    x_fit = np.linspace(np.deg2rad(-60), np.deg2rad(60), 100)
    
    for i in range(min(3, len(all_fits))):
        neuron_responses = responses[i, :]
        params = all_fits[i]
        
        # Plot raw data
        ax1.scatter(angles_rad, neuron_responses, alpha=0.5, label=f'Neuron {i+1}')
        
        # Plot fitted curve
        y_fit = von_mises(x_fit, *params)
        ax1.plot(x_fit, y_fit)
    
    ax1.set_xlabel('Angle (radians)')
    ax1.set_ylabel('Response')
    #ax1.set_title('Example Tuning Curves')
    ax1.legend()
    
    # Plot kappa distribution on polar plot
    ax2 = plt.subplot(122, projection='polar')
    
    # Get preferred directions (mu parameter from fits)
    preferred_directions = [params[2] for params in all_fits]  # mu is the third parameter
    
    # Create scatter plot
    ax2.scatter(preferred_directions, kappas)
    
    # Set the limits of the polar plot to only show the relevant angle range
    ax2.set_thetamin(-60)
    ax2.set_thetamax(60)
    
    #ax2.set_title(f'Kappa Values vs Preferred Direction\n(n={len(kappas)} neurons)')
    ax2.set_rticks([0, max(kappas)/2, max(kappas)])
    
    # Add summary statistics to the plot
    plt.figtext(0.02, 0.02, 
                f'Mean κ = {np.mean(kappas):.2f} ± {np.std(kappas):.2f}',
                fontsize=8)
    
    plt.tight_layout()
    plt.show()
    
    return fig

In [ ]:
master =  Path(r"Z:\Hagar and Ot\e0075\habenula")
fish_list = list(master.glob("*_f*"))


In [ ]:
n_neurons = np.shape(traces)[1]
angles = np.linspace(-60, 60, 8)  # 8 positions from -60 to 60 degrees


In [ ]:
fish = fish_list[15] / "suite2p"
planes = list(fish.glob("*00*"))
    
neural_data_groups_l = []
neural_data_groups_r = []
stimulus_data_groups = []

for plane in planes:
    
    traces = fl.load(plane / 'filtered_traces.h5')['undetr']
    regs = fl.load(plane / 'sensory_regressors_cells.h5')['regressors']

    hab_coords_l = fl.load(plane / 'habenula_coords.h5')['lhab_coords']
    hab_coords_r = fl.load(plane / 'habenula_coords.h5')['rhab_coords']

    habenula_traces_l = traces[:,hab_coords_l]
    habenula_traces_r = traces[:,hab_coords_r]
    
    neural_data_groups_l =  neural_data_groups_l + [habenula_traces_l]
    neural_data_groups_r =  neural_data_groups_r + [habenula_traces_r]
    
    stimulus_data_groups = stimulus_data_groups + [regs]
    
    


In [ ]:
# Lists to store results from all groups
all_kappas = []
all_fits = []
all_responses = []
all_r_squared = []
all_diagnostics = []

# Process each group
for group_idx, (neural_data, stimulus_data) in enumerate(zip(neural_data_groups_r, stimulus_data_groups)):
        print(f"Processing group {group_idx + 1}...")
        
        # Analyze this group
        kappas, fits, responses, r_squared, diagnostics = analyze_neural_data(neural_data, stimulus_data, angles)
        
        # Store results
        all_kappas.extend(kappas)
        all_fits.extend(fits)
        all_responses.append(responses)
        all_r_squared.extend(r_squared)
        all_diagnostics.extend(diagnostics)
    

# Combine responses from all groups
combined_responses = np.vstack(all_responses)

# Plot combined results
fig = plot_results(combined_responses, angles, all_kappas, all_fits, all_diagnostics)
    
# Print summary of fit quality
print("\nFit Quality Summary:")
n_total = len(all_diagnostics)
n_good_snr = sum(1 for d in all_diagnostics if d['signal_to_noise'] > 2)
n_good_peak = sum(1 for d in all_diagnostics if d['peak_angle_diff'] < 30)

print(f"Total neurons analyzed: {n_total}")
print(f"Neurons with good SNR: {n_good_snr} ({n_good_snr/n_total*100:.1f}%)")
print(f"Neurons with good peak alignment: {n_good_peak} ({n_good_peak/n_total*100:.1f}%)")


In [ ]:
file_name = 'von mises fit kappa r_hab.jpg'
fig.savefig(fish.parent / file_name, dpi=300)

file_name = 'von mises fit kappa r_hab.pdf'
fig.savefig(fish.parent / file_name, dpi=300)

In [ ]:
# Plot example neurons from different kappa ranges
plot_kappa_examples(combined_responses, angles, all_fits, all_r_squared)

# Find neurons with highest kappa values
highest_kappa_indices = np.argsort([params[1] for params in all_fits])[-5:]

print("\nNeurons with highest kappa values:")
for idx in highest_kappa_indices:
    kappa = all_fits[idx][1]
    r2 = all_r_squared[idx]
    print(f"Neuron {idx}: κ={kappa:.2f}, R²={r2:.2f}")

    # Plot detailed view of this neuron's responses
    # Note: You'll need to track which group each neuron belongs to
    # This is a simplified example assuming all neurons are from the first group
    plot_neuron_details(neural_data_groups_r[0], stimulus_data_groups[0], idx)

In [ ]:
np.shape(all_fits)